Step 1: Imports & Plotting Configuration

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import silhouette_samples

# Capstone package imports
from capstone.dataset import load_anime
from capstone.clustering import (
    DEFAULT_CLUSTER_FEATURES,
    evaluate_cluster_range,
    preprocess_clustering_features,
    run_clustering_pipeline,
)

# Visual styling
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["font.size"] = 11
plt.rcParams["figure.dpi"] = 120

# Create figures output directory if needed
FIGURES_DIR = Path("../../docs/figures")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Libraries imported successfully.")

ModuleNotFoundError: No module named 'matplotlib'

Step 2: Data Ingestion & Initial Feature Inspection

In [ ]:
# Load clean, processed catalog dataset
df_raw = load_anime()
print(f"Loaded anime dataset: {df_raw.shape[0]:,} titles across {df_raw.shape[1]} columns.\n")

# Display candidate clustering features summary
clustering_cols = [
    col for col in DEFAULT_CLUSTER_FEATURES if col in df_raw.columns
]
print(f"Selected continuous features: {clustering_cols}")
df_raw[clustering_cols].describe().T

Step 3: Feature Preprocessing & Scaling

In [ ]:
# Transform and standardize features (median imputation, log1p on skewed variables, StandardScaler)
X_scaled, scaler, feature_names = preprocess_clustering_features(df_raw)

print(f"Standardized feature matrix shape: {X_scaled.shape}")
print(f"Features: {feature_names}")
print(f"Mean per feature (should be ~0): {np.round(X_scaled.mean(axis=0), 4)}")
print(f"Standard deviation per feature (should be ~1): {np.round(X_scaled.std(axis=0), 4)}")

Step 4: Cluster Optimization Diagnostics — Elbow & Silhouette Evaluation

In [ ]:
# Evaluate Inertia and Silhouette across k in [2, 10]
k_range = range(2, 11)
eval_df = evaluate_cluster_range(X_scaled, k_range=k_range, random_state=42)

# Display tabular evaluation metrics
display(eval_df.style.highlight_max(subset=["silhouette_score"], color="lightgreen"))

Step 5: Visualization 1 — Elbow Curve & Silhouette Scores

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot Inertia (Elbow)
color = "tab:blue"
ax1.set_xlabel("Number of Clusters (k)", fontweight="bold")
ax1.set_ylabel("Inertia (Sum of Squared Distances)", color=color, fontweight="bold")
ax1.plot(eval_df["k"], eval_df["inertia"], marker="o", color=color, linewidth=2, markersize=8)
ax1.tick_params(axis="y", labelcolor=color)

# Plot Silhouette Score on secondary y-axis
ax2 = ax1.twinx()
color = "tab:orange"
ax2.set_ylabel("Average Silhouette Score", color=color, fontweight="bold")
ax2.plot(eval_df["k"], eval_df["silhouette_score"], marker="s", color=color, linewidth=2, linestyle="--", markersize=8)
ax2.tick_params(axis="y", labelcolor=color)

# Highlight k = 5 (strategic archetype baseline)
ax1.axvline(x=5, color="gray", linestyle=":", alpha=0.8, label="Selected Model (k=5)")
ax1.legend(loc="upper center")

plt.title("Cluster Optimization Diagnostics: Elbow Method & Silhouette Analysis", fontsize=14, fontweight="bold", pad=15)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "clustering_elbow_silhouette.png", dpi=300)
plt.show()

Step 6: Run Full Clustering Pipeline (k = 5)

In [ ]:
# Execute end-to-end pipeline with archetype labeling and 2D PCA projection
result = run_clustering_pipeline(df_raw, n_clusters=5, random_state=42, compute_pca=True)
df_clustered = result.data

print(f"Clustering complete. Silhouette Score: {result.silhouette:.4f} | Inertia: {result.inertia:,.2f}\n")

# Distribution of titles across strategic archetypes
cluster_dist = df_clustered["cluster_label"].value_counts().reset_index()
cluster_dist.columns = ["Strategic Archetype", "Anime Count"]
cluster_dist["Catalog Share (%)"] = np.round((cluster_dist["Anime Count"] / len(df_clustered)) * 100, 2)
display(cluster_dist)

Step 7: Centroid Analysis & Archetype Profiling

In [ ]:
# Inspect unscaled feature centroids per archetype
centroids = result.cluster_centroids.copy()
centroids["Archetype"] = [df_clustered[df_clustered["cluster_id"] == i]["cluster_label"].iloc[0] for i in centroids.index]
centroids = centroids.set_index("Archetype")

print("Unscaled Feature Centroids across Strategic Archetypes:")
display(centroids.round(2))

Step 8: Visualization 2 — 2D PCA Projection of Strategic Archetypes

In [ ]:
plt.figure(figsize=(11, 7))

# Custom color palette for the 5 archetypes
palette = {
    "Mainstream Hit": "#2ca02c",         # Green
    "Hidden Gem": "#1f77b4",             # Blue
    "High-Rated Niche": "#9467bd",       # Purple
    "Franchise Property": "#ff7f0e",     # Orange
    "Underperforming Catalog": "#d62728" # Red
}

ax = sns.scatterplot(
    data=df_clustered,
    x="pca_x",
    y="pca_y",
    hue="cluster_label",
    palette=palette,
    alpha=0.65,
    s=40,
    edgecolor="none"
)

# Compute explained variance if available
var_exp = result.pca_transformer.explained_variance_ratio_ * 100
plt.xlabel(f"Principal Component 1 ({var_exp[0]:.1f}% Variance)", fontweight="bold")
plt.ylabel(f"Principal Component 2 ({var_exp[1]:.1f}% Variance)", fontweight="bold")
plt.title("2D PCA Projection of Anime Consumer Archetypes (k = 5)", fontsize=14, fontweight="bold", pad=15)

plt.legend(title="Strategic Archetype", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=True)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pca_clusters_projection.png", dpi=300)
plt.show()

Step 9: Visualization 3 — Standardized Radar / Parallel Centroid Profiles

In [ ]:
# Calculate scaled mean centroids for normalized profile comparison
scaled_features_df = pd.DataFrame(X_scaled, columns=feature_names)
scaled_features_df["cluster_label"] = df_clustered["cluster_label"].values
norm_centroids = scaled_features_df.groupby("cluster_label").mean()

# Plot heatmapped feature signatures
plt.figure(figsize=(9, 5))
sns.heatmap(
    norm_centroids,
    annot=True,
    cmap="vlag",
    center=0,
    fmt=".2f",
    linewidths=0.8,
    cbar_kws={"label": "Standardized Deviation from Mean (Z-score)"}
)

plt.title("Normalized Centroid Feature Signatures by Archetype", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Feature Name", fontweight="bold")
plt.ylabel("Strategic Archetype", fontweight="bold")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "centroid_heatmap_signatures.png", dpi=300)
plt.show()

Step 10: Candidate Mining for Final Acquisition Recommendations

In [ ]:
# Extract sample flagship titles representing each archetype for team review
print("=" * 80)
print("SAMPLE FLAGSHIP TITLES BY ARCHETYPE")
print("=" * 80)

for archetype in df_clustered["cluster_label"].unique():
    subset = df_clustered[df_clustered["cluster_label"] == archetype]
    
    # Sort by score or popularity
    if archetype == "Hidden Gem":
        sample_titles = subset.sort_values(by="score", ascending=False)[["title", "score", "popularity", "n_favorites"]].head(5)
    elif archetype == "Mainstream Hit":
        sample_titles = subset.sort_values(by="n_favorites", ascending=False)[["title", "score", "popularity", "n_favorites"]].head(5)
    elif archetype == "Franchise Property":
        sample_titles = subset.sort_values(by="episodes", ascending=False)[["title", "score", "episodes", "n_favorites"]].head(5)
    else:
        sample_titles = subset.sort_values(by="score", ascending=False)[["title", "score", "popularity"]].head(5)
        
    print(f"\n>>> Archetype: {archetype} (Total Titles: {len(subset):,})")
    display(sample_titles)